# InferencePipeline - MPR model, end-to-end SBI

This notebook walks through the complete `InferencePipeline` workflow:

1. Set up an MPR simulation on the 84-node Hagmann connectome
2. Run a parameter sweep with the **numba** backend
3. Train a MAF density estimator (1-round SNPE)
4. Inspect the posterior with a pairplot
5. Cache raw recordings and re-extract with different features
6. Save / load a checkpoint and resume

**Runtime:** ~5–10 min on a modern CPU (numba backend, 300 simulations).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from vbi.simulator import Simulator
from vbi.simulator.spec import (
    SimulationSpec, IntegratorSpec, CouplingSpec, MonitorSpec,
    prepare_connectivity, save_connectivity,
)
from vbi.simulator.models.mpr import mpr
from vbi.feature_extraction import (
    FeaturePipeline, get_features_by_domain, get_features_by_given_names,
)
from vbi.inference import InferencePipeline, SNPE, MAF, TrainingOptions, BoxUniform

# docs/examples/data/ has weights.txt + tract_lengths.txt for 84-node connectome
DATA_DIR = Path('../data')

# All output files go here (created automatically)
OUT_DIR = Path('outputs')
OUT_DIR.mkdir(exist_ok=True)

## 1 - Load connectivity and build SimulationSpec

`prepare_connectivity` accepts **.txt**, **.csv**, **.npy**, **.npz**, or plain numpy arrays -
bring your own data by replacing the paths below.

In [ ]:
# ── Option A: load from the bundled 84-node text files in docs/examples/data/ ─
W, D = prepare_connectivity(
    weights       = DATA_DIR / 'weights.txt',
    tract_lengths = DATA_DIR / 'tract_lengths.txt',
    normalize     = True,   # row-sum normalise (recommended for MPR/VEP)
)

# ── Option B: bring your own numpy arrays ─────────────────────────────────────
# W_custom = np.load('my_sc_matrix.npy')     # (n, n)
# W, D = prepare_connectivity(W_custom)      # tract_lengths=None → zero delays

# ── Option C: load from a .mat file (use scipy first) ─────────────────────────
# from scipy.io import loadmat
# mat = loadmat('my_data.mat')
# W, D = prepare_connectivity(mat['weights'], mat['tract_lengths'])

n_nodes = W.shape[0]
print(f'Connectivity: {n_nodes} nodes   W range [{W.min():.4f}, {W.max():.4f}]')

# Save as .npz in outputs/ - used by from_config in cell 8
save_connectivity(W, D, OUT_DIR / 'connectivity_84.npz', normalize=False)

In [ ]:
sim_spec = SimulationSpec(
    model      = mpr,
    integrator = IntegratorSpec(method='heun', dt=0.1),
    coupling   = CouplingSpec('linear', a=1.0),
    monitors   = (MonitorSpec('tavg', period=1.0),),
    weights    = W,
    tract_lengths = D,
    speed      = 4.0,
    node_params = {
        'eta': np.full(n_nodes, -4.6),
    },
)

## 2 - Define prior and feature pipeline

In [ ]:
# Infer G (global coupling) and η (mean excitability)
prior = BoxUniform(
    low  = np.array([0.5, -5.5]),
    high = np.array([4.0, -3.0]),
    param_names = ['G', 'eta'],
)

# FC matrix (upper triangle) as features
cfg = get_features_by_domain('connectivity')
cfg = get_features_by_given_names(cfg, ['calc_fc'])
pipeline_fc = FeaturePipeline(cfg, signal='tavg', t_cut=500.0)

# Quick single-run to check feature labels
result = Simulator(sim_spec, backend='numpy').run(600.0)
labels, values = pipeline_fc.extract(result)
print(f'n_features={len(labels)}  first 5: {labels[:5]}')

## 3 - Build `InferencePipeline` and run 1-round SNPE

In [ ]:
inf = InferencePipeline(
    sim_spec          = sim_spec,
    prior             = prior,
    feature_pipeline  = pipeline_fc,
    integrator_backend = 'numba',   # parallel sweep
    engine            = SNPE(prior, density_estimator=MAF(backend='auto')),  # JAX if available, else numpy
    training          = TrainingOptions(batch_size=128, stop_after_epochs=30, max_epochs=200),
)
print(inf)

In [ ]:
%%time
# Round 1: 300 simulations from prior
theta, x = inf.simulate(
    num_simulations = 300,
    duration        = 2000.0,   # ms
    seed            = 0,
)
print(f'theta: {theta.shape}   x: {x.shape}')

In [ ]:
%%time
estimator = inf.train()   # uses the TrainingOptions passed at construction
print('Training done.')

In [ ]:
inf.plot_loss()

## 4 - Build posterior and visualise

In [ ]:
posterior = inf.build_posterior(estimator)

# Use the first simulated feature vector as a synthetic observation
x_obs = x[0]

samples = posterior.sample((1000,), x=x_obs)
print(f'Posterior samples: {samples.shape}   (should be (1000, 2))')

In [ ]:
inf.pairplot(x_obs, num_samples=1000)

## 5 - Simulation cache: re-extract features without re-simulating

For long simulations (CUDA or large networks), caching raw recordings to disk
lets you change the feature pipeline later without re-running the sweeper.

In [ ]:
%%time
CACHE_DIR = OUT_DIR / 'sim_cache_mpr_r1'

inf2 = InferencePipeline(
    sim_spec=sim_spec, prior=prior, feature_pipeline=pipeline_fc,
    integrator_backend='numba',
)

theta_c, x_fc = inf2.simulate(
    num_simulations = 200,
    duration        = 2000.0,
    seed            = 1,
    cache_dir       = CACHE_DIR,
    chunk_size      = 100,   # 100 sims per .npz chunk
)
print(f'Cached to {CACHE_DIR}. Files:')
for f in sorted(CACHE_DIR.iterdir()): print(' ', f.name)

In [ ]:
# Re-extract with statistical features - no simulation needed
cfg_stat = get_features_by_domain('statistical')
cfg_stat = get_features_by_given_names(cfg_stat, ['calc_mean', 'calc_std'])
pipeline_stat = FeaturePipeline(cfg_stat, signal='tavg', t_cut=500.0)

theta_re, x_stat = InferencePipeline.extract_from_cache(CACHE_DIR, pipeline_stat)
print(f'Re-extracted: theta {theta_re.shape}  x_stat {x_stat.shape}')
print('FC features shape:   ', x_fc.shape)
print('Stat features shape: ', x_stat.shape)

## 6 - Save checkpoint and reload

In [ ]:
inf.save(OUT_DIR / 'vbi_mpr_checkpoint.npz')
print('Saved.')

# Reload - must supply sim_spec and feature_pipeline again
inf_loaded = InferencePipeline.load(
    OUT_DIR / 'vbi_mpr_checkpoint.npz',
    sim_spec         = sim_spec,
    feature_pipeline = pipeline_fc,
    prior            = prior,
)
print(inf_loaded)

# Verify posterior still works
post2    = inf_loaded.build_posterior()
samples2 = post2.sample((500,), x=x_obs)
print(f'Post-load samples: {samples2.shape}')

## 7 - Round 2: posterior-focused simulation (sequential SNPE)

In [ ]:
%%time
# Simulate from the round-1 posterior to focus budget on high-posterior regions
theta_r2, x_r2 = inf.simulate(
    num_simulations = 200,
    duration        = 2000.0,
    proposal        = posterior,   # sample from posterior, not prior
    x_obs           = x_obs,
    seed            = 2,
)
print(f'Round 2 data: theta {theta_r2.shape}  x {x_r2.shape}')
print(f'Total simulations so far: {inf._snpe.n_simulations}')

In [ ]:
%%time
est_r2   = inf.train(training_batch_size=128, stop_after_epochs=30, max_num_epochs=200)
post_r2  = inf.build_posterior(est_r2)
samples_r2 = post_r2.sample((1000,), x=x_obs)
print(f'Round-2 samples: {samples_r2.shape}')

In [ ]:
# Compare round-1 vs round-2 posteriors
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, samp, label in zip(
    axes,
    [samples, samples_r2],
    ['Round 1 (prior)', 'Round 2 (posterior-focused)'],
):
    ax.scatter(samp[:, 0], samp[:, 1], alpha=0.3, s=5)
    ax.set_xlabel('G')
    ax.set_ylabel('η')
    ax.set_title(label)
    ax.axvline(theta[0, 0], color='red', ls='--', label='x_obs param')
    ax.axhline(theta[0, 1], color='red', ls='--')
    ax.legend()
plt.tight_layout()

## 8 - from_config: reproducible workflow via YAML

The same workflow can be driven from a config file for reproducibility.

In [ ]:
import json

config = {
    'sim': {
        'model': 'mpr',
        'connectivity': str(OUT_DIR / 'connectivity_84.npz'),  # saved in cell 3
        'node_params': {'eta': -4.6},
        'dt': 0.1,
        'method': 'heun',
        'monitors': [{'kind': 'tavg', 'period': 1.0}],
        'coupling': {'kind': 'linear', 'a': 1.0},
        'speed': 4.0,
    },
    'prior': {
        'type': 'BoxUniform',
        'low':  [0.5, -5.5],
        'high': [4.0, -3.0],
        'param_names': ['G', 'eta'],
    },
    'pipeline': {
        'features': ['calc_fc'],
        'signal':   'tavg',
        't_cut':    500.0,
    },
    'inference': {
        'density_estimator': 'maf',
        'integrator_backend': 'numba',
        'estimator_backend': 'auto',
        'training': {
            'training_batch_size': 128,
            'stop_after_epochs': 30,
            'max_num_epochs': 200,
        },
    },
}

config_path = OUT_DIR / 'vbi_mpr_config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f'Config saved to {config_path}')

inf_cfg = InferencePipeline.from_config(config)   # pass dict directly (no file read needed)
print(inf_cfg)
print('Default train kwargs:', inf_cfg._default_train_kwargs)